# 20. Validation Test (Pipeline 01)

- Goal: run structural validation on the loaded pose CSV.
- Docs: `docs_eng/pipeline/01_validation.md` / `docs/pipeline/01_validation.md`
- Inputs: Default pose CSV path from the input cell.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Structural pass/fail report and data-quality warning provenance.


In [ ]:
import pandas as pd

from movement.config import (
    make_coordinate_columns,
    make_required_columns,
    make_visibility_columns,
)
from movement.io import load_pose_csv
from movement.stage_context import find_project_root
from movement.validation import run_basic_validation

PROJECT_ROOT = find_project_root()
pose_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
df = load_pose_csv(pose_path)

report = run_basic_validation(
    df=df,
    required_columns=make_required_columns(),
    coordinate_columns=make_coordinate_columns(),
    visibility_columns=make_visibility_columns(),
)

structural_checks = {
    "required_columns": report["required_columns"]["passed"],
    "frame_continuity": report["frame_continuity"]["passed"],
    "timestamp": report["timestamp"]["passed"],
    "missing_values": report["missing_values"]["passed"],
}
structural_passed = all(structural_checks.values())
visibility_passed = report.get("visibility", {}).get("passed", None)

validation_status = "passed"
if structural_passed and visibility_passed is False:
    validation_status = "passed_with_visibility_warnings"
elif not structural_passed:
    validation_status = "structural_failed"

summary = {
    "validation_status": validation_status,
    "top_level_passed": report["passed"],
    "structural_passed": structural_passed,
    "visibility_passed": visibility_passed,
    "estimated_fps": report["timestamp"].get("estimated_fps"),
    "num_rows": len(df),
    "start_frame": report["frame_continuity"].get("start_frame"),
    "end_frame": report["frame_continuity"].get("end_frame"),
    "num_missing_frames": report["frame_continuity"].get("num_missing_frames"),
    "num_duplicated_frames": report["frame_continuity"].get("num_duplicated_frames"),
}

print("pose:", pose_path.relative_to(PROJECT_ROOT))
summary


In [ ]:
assert structural_passed, "Structural validation failed; inspect required columns, frames, timestamps, or missing values before continuing."

visibility_report = report.get("visibility", {})
low_visibility = visibility_report.get("low_visibility_ratio_by_column", {})
low_visibility_table = (
    pd.Series(low_visibility, name="low_visibility_ratio")
    .sort_values(ascending=False)
    .to_frame()
)

print("Structural validation passed.")
if visibility_passed is False:
    print("Visibility warning is present for p01; keep this as data-quality provenance for reliability gates.")
else:
    print("Visibility validation passed or was not assessed.")

display(low_visibility_table.head(12))